# Nutzung von Raspberry Pi GPIO mit C/C++

Dieses Notebook soll als C/C++-Alternative des RPi.GPIO Notebooks (LED_Projekt) dienen.

Dazu machen wir Gebrauch der *pigpio*-Library (https://abyz.me.uk/rpi/pigpio/)

Dokumentation aller Funktionen: https://abyz.me.uk/rpi/pigpio/cif.html
<br></br>

## Installation von *pigpiod*

Um die Bibliothek zu nutzen, müssen wir den *pigpio daemon* (pigpiod) herunterladen und installieren (https://abyz.me.uk/rpi/pigpio/download.html):

In [ ]:
!wget https://github.com/joan2937/pigpio/archive/master.zip
!unzip master.zip
!make -C pigpio-master
!sudo make -C pigpio-master install

Der Daemon sollte normalerweise automatisch starten, falls Sie ein Programm, die Gebrauch von `pigpio.h` macht, ausführen.

## Nutzung von *pigpio* in C/C++

Um die Bibliothek jetzt Nutzen zu können, müssen wir die `pigpio.h`-Datei in unserem C/C++-Projekt inkludieren:



```
#include <pigpio.h>

int main() { ... }
```

<br></br>
Um unser Programm dann zu kompilieren, müssen wir dem *Compiler* die Bibliothek als Parameter übergeben:



```
gcc meinCode.c -o meinCode -lpipgio
```



## Blinkende LED

Alle Funktionen der *pigpio*-Bibliothek liefern einen negativen Rückgabewert, im Falle eines Fehlers. Die Pins werden mit ihrer *Broadcom*-Nummer identifiziert.

Um eine LED blinken zu lassen, würde der Quellcode beispielsweise so aussehen:

In [ ]:
%%writefile blinkingled.cpp

#include <pigpio.h>
#include <signal.h>
#include <iostream>
#include <unistd.h>

#define LED_PIN 17

volatile sig_atomic_t terminate = 0;

void sigintHandler(int signum) {
  terminate = 1;
}


int main() {

  // Initialisierung der GPIOs
  if(gpioInitialise() < 0) {
    std::cout << "GPIO konnte nicht initialisiert werden.\n";
    return PI_INIT_FAILED;
  }

  // Pin auf PI_INPUT oder PI_OUTPUT setzen
  gpioSetMode(LED_PIN, PI_OUTPUT);

  // Falls CTRL-C gedrückt wird, soll Programm sauber beendet werden
  signal(SIGINT, sigintHandler);
  std::cout << "CTRL-C zum beenden.\n";

  while(!terminate) {

    gpioWrite(LED_PIN, PI_HIGH);        // Pin 17 auf HIGH Pegel setzen
    gpioDelay(1000000);                 // 1 Sekunde schlafen
    gpioWrite(LED_PIN, PI_LOW);         // Pin 17 auf LOW Pegel setzen
    gpioDelay(1000000);                 // 1 Sekunde schlafen

  }

  // Clean up nach CTRL-C
  gpioWrite(LED_PIN, PI_LOW);
  gpioSetMode(LED_PIN, PI_INPUT);
  gpioTerminate();
  std::cout << "\n";


  return 0;

}

In [ ]:
!g++ blinkingled.cpp -o blinkingled -lpigpio
!sudo ./blinkingled

2 LEDs ein- und ausschalten:

In [ ]:
%%writefile blinkingtwoled.cpp

#include <pigpio.h>
#include <signal.h>
#include <iostream>

#define LED1_PIN 17
#define LED2_PIN 27

volatile sig_atomic_t terminate = 0;

void sigintHandler(int signum) {
  terminate = 1;
}


int main() {

  // Initialisierung der GPIOs
  if(gpioInitialise() < 0) {
    std::cout << "GPIO konnte nicht initialisiert werden.\n";
    return PI_INIT_FAILED;
  }

  // Pins auf PI_INPUT oder PI_OUTPUT setzen
  gpioSetMode(LED1_PIN, PI_OUTPUT);
  gpioSetMode(LED2_PIN, PI_OUTPUT);


  // Falls CTRL-C gedrückt wird, soll Programm sauber beendet werden
  signal(SIGINT, sigintHandler);
  std::cout << "CTRL-C zum beenden.\n";

  while(!terminate) {

    gpioWrite(LED1_PIN, PI_HIGH);       // Pin 17 auf HIGH Pegel setzen
    gpioWrite(LED2_PIN, PI_LOW);        // Pin 27 auf LOW Pegel setzen
    gpioDelay(1000000);                 // 1 Sekunde schlafen
    gpioWrite(LED1_PIN, PI_LOW);        // Pin 17 auf LOW Pegel setzen
    gpioWrite(LED2_PIN, PI_HIGH);       // Pin 27 auf HIGH Pegel setzen
    gpioDelay(1000000);                 // 1 Sekunde schlafen

  }

  // Clean up nach CTRL-C
  gpioWrite(LED1_PIN, PI_LOW);
  gpioWrite(LED2_PIN, PI_LOW);
  gpioSetMode(LED1_PIN, PI_INPUT);
  gpioSetMode(LED2_PIN, PI_INPUT);
  gpioTerminate();

  std::cout << "\n";



  return 0;

}

In [ ]:
!g++ blinkingtwoled.cpp -o blinkingtwoled -lpigpio
!sudo ./blinkingtwoled

## LED Helligkeit mit PWM steuern

Mit PWM (https://de.wikipedia.org/wiki/Pulsdauermodulation) kann man die Helligkeit einer LED einstellen.

Hier ein kleines Beispiel, was eine LED zum pulsieren bringt:

In [ ]:
%%writefile pwm.cpp

#include <iostream>
#include <signal.h>
#include <unistd.h>
#include <pigpio.h>

#define LED_PIN 17

volatile sig_atomic_t terminate = 0;

void sigintHandler(int signum) {
    terminate = 1;
}


int main() {

    if(gpioInitialise() < 0) {
      std::cout << "GPIO konnte nicht initialisiert werden.\n";
      return PI_INIT_FAILED;
    }

    gpioSetMode(LED_PIN, PI_OUTPUT);

    signal(SIGINT, sigintHandler);

    std::cout << "CTRL-C zum beenden.\n";

    int duty = 0;
    int step = 5;

    while(!terminate) {


        // Der Duty-Cycle geht standardgemäß von 0 - 255 (https://abyz.me.uk/rpi/pigpio/cif.html#gpioPWM)
        // Duty von 255 -> 100% Helligkeit
        // Duty von 128 -> 50% Helligkeit
        // Duty von 0 -> 0% Helligkeit (aus)
        gpioPWM(LED_PIN, duty);

        gpioDelay(10000);

        duty += step;

        if(duty >= 255) {       // LED wird immer dunkler
            duty = 255;
            step = -step;
        } else if(duty <= 0) {  // LED wird immer heller
            duty = 0;
            step = -step;
        }

    }

    gpioPWM(LED_PIN, 0);
    gpioWrite(LED_PIN, PI_LOW);
    gpioSetMode(LED_PIN, PI_INPUT);
    gpioTerminate();

    std::cout << "\n";

    return 0;

}

In [ ]:
!g++ pwm.cpp -o pwm -lpigpio
!sudo ./pwm

## I²C Geräte

Die *pigpio*-Bibliothek unterstützt auch Kommunikation über den I²C-Bus.

Dies ist nützlich, um z.B. Sensorwerte von einem Accelerometer auszulesen, der über I²C-Bus kommuniziert.

Hier ein Beispiel, wie man Beschleunigungswerte von einem MPU-9250 Accelerometer auslesen kann:

In [ ]:
%%writefile accel.cpp

#include <pigpio.h>
#include <iostream>
#include <signal.h>
#include <unistd.h>
#include <math.h>

// I2C Adresse des Geräts
const uint8_t MPU9250_ADDR = 0x68;

// Register definieren, Daten aus: https://invensense.tdk.com/wp-content/uploads/2015/02/RM-MPU-9250A-00-v1.6.pdf
const uint8_t PWR_MGMT_1 = 0x6B;
const uint8_t PWR_MGMT_2 = 0x6C;

const uint8_t ACCEL_XOUT_H = 0x3B;
const uint8_t ACCEL_XOUT_L = 0x3C;

const uint8_t ACCEL_YOUT_H = 0x3D;
const uint8_t ACCEL_YOUT_L = 0x3E;

const uint8_t ACCEL_ZOUT_H = 0x3F;
const uint8_t ACCEL_ZOUT_L = 0x40;

const uint8_t ACCEL_CONFIG = 0x1C;


volatile sig_atomic_t terminate = 0;

void sigintHandler(int signum) {
    terminate = 1;
}



int main() {

  if(gpioInitialise() < 0) {
    std::cerr << "GPIO konnte nicht initialisiert werden.\n";
    return PI_INIT_FAILED;
  }

  // Gerät wird geöffnet und mit dem Handle kann man auf das Gerät zugreifen
  // Paramater:
  // 1 -> Bus Nr. 1 auf dem Pi wird genutzt
  // MPU9250_ADDR -> Adresse des Accelerometer
  // 0 -> Flags (ungenutzt)
  int handle = i2cOpen(1, MPU9250_ADDR, 0);

  if (handle < 0) {
    std::cerr << "Konnte Gerät nicht öffnen.\n";
    return handle;
  }

  std::cout << "I2C Gerät erfolgreich geöffnet mit Nummer: " << handle << "\n";
  gpioDelay(1000000);

  signal(SIGINT, sigintHandler);


  // Accelerometer einschalten
  if(i2cWriteByteData(handle, PWR_MGMT_1,0x00) < 0) {
    std::cerr << "Konnte Accelerometer nicht einschalten. Beende...\n";
    return -1;

  }

  // Standardkonfiguration
  i2cWriteByteData(handle, ACCEL_CONFIG, 0x00);

  system("clear");

  while(!terminate) {

    // Lesen der Beschleunigungsregistern
    int16_t rawX = ((i2cReadByteData(handle,ACCEL_XOUT_H) << 8) | i2cReadByteData(handle,ACCEL_XOUT_L));
	  int16_t rawY = ((i2cReadByteData(handle,ACCEL_YOUT_H) << 8) | i2cReadByteData(handle,ACCEL_YOUT_L));
	  int16_t rawZ = ((i2cReadByteData(handle,ACCEL_ZOUT_H) << 8) | i2cReadByteData(handle,ACCEL_ZOUT_L));

    // Beschleunigung in g
    float accelX = rawX / 16384.0f;
    float accelY = rawY / 16384.0f;
    float accelZ = rawZ / 16384.0f;

    // Gesamtbeschleunigung ausrechnen
    float acceleration = std::sqrt(accelX * accelX + accelY * accelY + accelZ * accelZ);


    printf("\033[1;1H");
    std::cout << "Aktuelle Beschleunigung: " << acceleration << " g      \n";
    fflush(stdout);


    gpioDelay(1000000);

  }

  i2cClose(handle);
  gpioTerminate();
  std::cout << "\n";


  return 0;

}


In [ ]:
!g++ accel.cpp -o accel -lpigpio
!sudo ./accel

Beispielcode, um die Temperatur eines BME280-Sensors auszulesen:

In [ ]:
%%writefile bme280.cpp

#include <pigpio.h>
#include <iostream>
#include <csignal>
#include <unistd.h>
#include <cstdint>

static const uint8_t BME280_ADDR = 0x77;

// BME280-Register
static const uint8_t REG_ID        = 0xD0;
static const uint8_t REG_RESET     = 0xE0;
static const uint8_t REG_CALIB_START = 0x88;
static const uint8_t REG_CTRL_MEAS = 0xF4;
static const uint8_t REG_TEMP_MSB  = 0xFA;

volatile sig_atomic_t terminate = 0;

void sigintHandler(int sig) {
    terminate = 1;
}

// Kalibrierungsdaten Struktur
struct BME280_Calib {
    uint16_t dig_T1;
    int16_t  dig_T2;
    int16_t  dig_T3;
};

int main() {

    if (gpioInitialise() < 0) {
        std::cerr << "GPIO Initialisierung fehlgeschlagen\n";
        return 1;
    }

    int handle = i2cOpen(1, BME280_ADDR, 0);

    if (handle < 0) {
        std::cerr << "Fehler: Konnte I2C Gerät nicht öffnen\n";
        gpioTerminate();
        return -1;
    }

    std::cout << "I2C geöffnet, Handle = " << handle << "\n";

    signal(SIGINT, sigintHandler);

    // Chip-ID prüfen
    int id = i2cReadByteData(handle, REG_ID);

    if (id < 0) {
        std::cerr << "Fehler: konnte Chip-ID nicht lesen\n";
        i2cClose(handle);
        gpioTerminate();
        return -1;
    }

    if (id != 0x60) {
        std::cerr << "Kein BME280 erkannt (ID != 0x60, ID = 0x"
                  << std::hex << id << std::dec << ")\n";
        i2cClose(handle);
        gpioTerminate();
        return -1;
    }

    std::cout << "BME280 erkannt (ID = 0x" << std::hex << id << std::dec << ")\n";

    // Kalibrierungsdaten lesen
    char calibBuf[6];

    int res = i2cReadI2CBlockData(handle, REG_CALIB_START, calibBuf, 6);

    if (res != 6) {
        std::cerr << "Fehler: konnte Kalibrierungsdaten nicht lesen\n";
        i2cClose(handle);
        gpioTerminate();
        return -1;
    }

    BME280_Calib calib;

    calib.dig_T1 = (uint16_t)(((uint8_t)calibBuf[0]) | ((uint8_t)calibBuf[1] << 8));
    calib.dig_T2 = (int16_t)(((uint8_t)calibBuf[2]) | ((uint8_t)calibBuf[3] << 8));
    calib.dig_T3 = (int16_t)(((uint8_t)calibBuf[4]) | ((uint8_t)calibBuf[5] << 8));

    std::cout << "Kalibrierung geladen: T1=" << calib.dig_T1
              << " T2=" << calib.dig_T2
              << " T3=" << calib.dig_T3 << "\n";

    // Sensor konfigurieren: Temp-Oversampling x1, Druck ignoriert, Mode = forced
    // ctrl_meas: osrs_t = 001, osrs_p = 000, mode = 01 → 0b00100001 = 0x21
    uint8_t ctrl_meas = 0x21;

    if (i2cWriteByteData(handle, REG_CTRL_MEAS, ctrl_meas) < 0) {
        std::cerr << "Fehler: Sensor Konfiguration fehlgeschlagen\n";
        i2cClose(handle);
        gpioTerminate();
        return -1;
    }

    while (!terminate) {

        // Messung starten (erneut forced-Mode schreiben)
        if (i2cWriteByteData(handle, REG_CTRL_MEAS, ctrl_meas) < 0) {
            std::cerr << "Fehler: Messung starten fehlgeschlagen\n";
            break;
        }

        // Warte für Messung (je nach Oversampling etwas warten)
        usleep(50 * 1000);  // 50 ms — ggf. anpassen

        // Rohdaten Temperatur lesen
        char t_raw_buf[3];

        res = i2cReadI2CBlockData(handle, REG_TEMP_MSB, t_raw_buf, 3);

        if (res != 3) {
            std::cerr << "Fehler: Temperatur-Rohdaten konnten nicht gelesen werden\n";
            break;
        }

        int32_t adc_T = (((uint8_t)t_raw_buf[0] << 12) |
                         ((uint8_t)t_raw_buf[1] << 4) |
                         ((uint8_t)t_raw_buf[2] >> 4));

        // Temperatur-Kompensation (laut Datenblatt)
        int32_t var1 = ((((adc_T >> 3) - ((int32_t)calib.dig_T1 << 1))) * (int32_t)calib.dig_T2) >> 11;
        int32_t var2 = (((((adc_T >> 4) - (int32_t)calib.dig_T1) * ((adc_T >> 4) - (int32_t)calib.dig_T1)) >> 12) * (int32_t)calib.dig_T3) >> 14;
        int32_t t_fine = var1 + var2;
        float T  = (t_fine * 5 + 128) >> 8;
        float temperature = T / 100.0f;

        std::cout << "Temperatur: " << temperature << " °C\n";

        sleep(1);
    }

    i2cClose(handle);
    gpioTerminate();
    std::cout << "Programm beendet.\n";
    return 0;
}

In [ ]:
!g++ bme280.cpp -o bme280 -lpigpio
!sudo ./bme280